# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [1]:
import os
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. قراءة المفاتيح السرية من Kaggle Secrets
user_secrets = UserSecretsClient()
openrouter_key = user_secrets.get_secret("OPENROUTER_API_KEY")
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")

# 2. تهيئة عميل OpenRouter (للنماذج الرخيصة)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

# 3. تهيئة عميل Inception Labs (لنموذج Mercury)
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key,
)

print("✅ تم الاتصال بالمنصتين بنجاح!")
print("📡 OpenRouter: جاهز للنماذج الرخيصة")
print("📡 Inception Labs: جاهز لـ Mercury")

✅ تم الاتصال بالمنصتين بنجاح!
📡 OpenRouter: جاهز للنماذج الرخيصة
📡 Inception Labs: جاهز لـ Mercury


In [ ]:


try:
    response = inception_client.chat.completions.create(
        model="mercury-2.5",
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ],
        max_tokens=500,
        reasoning_effort="low"
    )
    
    content = response.choices[0].message.content
    
    if content:
        print(f"✅ نجح Mercury! الرد: '{content.strip()}'")
        print(f"\n📊 إحصائيات الرد:")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - prompt_tokens: {response.usage.prompt_tokens}")
            print(f"   - completion_tokens: {response.usage.completion_tokens}")
            print(f"   - total_tokens: {response.usage.total_tokens}")
    else:
        print("⚠️ المحتوى لا يزال فارغاً")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - total_tokens المستهلكة: {response.usage.total_tokens}")
    
except Exception as e:
    print(f"❌ فشل! السبب: {str(e)[:200]}")

In [2]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import traceback

print(f"GPU: {torch.cuda.get_device_name(0)}")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

models_to_check = [
    {"name": "Qwen/Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LiquidAI/LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

print("🔍 بدء اختبار النماذج محلياً على GPU...\n")

for model_info in models_to_check:
    print(f"⏳ جاري تحميل: {model_info['name']}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )

        messages = [{"role": "user", "content": "Say hello in one sentence."}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            return_tensors="pt", return_dict=True
        ).to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=50)
        input_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        print(f"   ✅ نجح! الرد: '{content.strip()[:50]}'\n")

        del model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"   ❌ فشل! نوع الخطأ: {type(e).__name__}")
        traceback.print_exc()
        print()

print("✅ انتهى الاختبار!")

GPU: Tesla T4
🔍 بدء اختبار النماذج محلياً على GPU...

⏳ جاري تحميل: Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

   ✅ نجح! الرد: '<think>
Okay, the user wants me to say hello in on'

⏳ جاري تحميل: LiquidAI/LFM2.5-2.6B


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

   ✅ نجح! الرد: 'The user wants me to say hello in one sentence. Th'

✅ انتهى الاختبار!
